# L2C — Local exploration and ML benchmark

This notebook calls the working repository modules. Inventory, crop preparation and optional ML benchmarking are implemented. Annex A extraction, element matching and PDF reporting remain pending.

Keep confidential data outside OneDrive and Git. Launch Jupyter with the supplied configuration: outputs are cleared on every save. Never paste confidential data into cell source or Markdown.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys
REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
assert (REPO / "l2c").is_dir(), "Launch from the repository root or notebooks directory."
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
DATA_DIR = Path(r"C:\Users\Admin\Downloads\l2c-participants")
OUTPUT_DIR = Path(r"C:\Users\Admin\Documents\Codex\l2c-local\notebook")
PROJECT = "EspCa3B"
PAGE = 37
RUN_ML = False
assert DATA_DIR.is_dir(), "Set DATA_DIR to the local dataset."
for path in (DATA_DIR.resolve(), OUTPUT_DIR.resolve()):
    assert not path.is_relative_to(REPO), "Keep data outside the repository."
    assert not any("onedrive" in part.lower() for part in path.parts), "Use a folder outside OneDrive."
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Configuration ready. ML enabled:", RUN_ML)


## 1. Explore the dataset

Inventory page counts and spreadsheet dimensions. This checks readability, not extraction accuracy.


In [ ]:
from l2c.inspect import inspect_dataset
inventory = inspect_dataset(DATA_DIR)
(OUTPUT_DIR / "inventory.json").write_text(json.dumps(inventory, indent=2), encoding="utf-8")
for project in inventory["projects"]:
    errors = sum("error_type" in f for f in project["files"])
    print(project["project"], "PDFs:", project["pdf_count"], "pages:", project["page_count"], "read errors:", errors)
print("Spreadsheets:", len(inventory["spreadsheets"]))


## 2. Prepare identical image crops

Three crops are selected around native bar-designation candidates. This biases the test toward text-bearing regions. Native text stays local for comparison; the models receive image crops only. These are not validated Annex A element records.


In [ ]:
PLAN = DATA_DIR / PROJECT / f"L2C_PLAN_STR_{PROJECT}.pdf"
SAMPLES_DIR = OUTPUT_DIR / PROJECT / "samples"
subprocess.run([sys.executable, "-m", "l2c.prepare_samples", "--pdf", str(PLAN),
                "--page", str(PAGE), "--output-dir", str(SAMPLES_DIR), "--count", "3"],
               cwd=REPO, check=True)
samples = json.loads((SAMPLES_DIR / "samples.json").read_text(encoding="utf-8"))
print("Prepared crops:", len(samples))


## 3. Optional local ML inference

Install requirements-ml.txt and CPU PyTorch first. Set RUN_ML=True to execute. Florence requires downloaded model weights. PaddleOCR downloads public weights if the cache is empty; image inference stays local. Qwen was not run on the tested computer because available memory was insufficient for the proposed configuration.


In [ ]:
MODEL_DIR = OUTPUT_DIR / "models" / "florence"
model_env = os.environ.copy()
model_env.update({"HF_HOME": str(OUTPUT_DIR / "hf-cache"), "HF_HUB_DISABLE_TELEMETRY": "1",
                  "PADDLE_PDX_CACHE_HOME": str(OUTPUT_DIR / "paddle-cache"),
                  "PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK": "True", "OMP_NUM_THREADS": "1"})
if RUN_ML:
    for backend in ("paddle", "florence"):
        command = [sys.executable, "-m", "l2c.ml_benchmark", "--backend", backend,
                   "--samples", str(SAMPLES_DIR / "samples.json"),
                   "--output", str(SAMPLES_DIR / f"{backend}-results.json")]
        if backend == "florence":
            assert MODEL_DIR.is_dir(), "Set MODEL_DIR to downloaded Florence weights."
            command += ["--model-dir", str(MODEL_DIR)]
        subprocess.run(command, cwd=REPO, env=model_env, check=True)
else:
    print("ML inference skipped. Enable RUN_ML after installing dependencies and weights.")


## 4. Compare saved outputs without displaying confidential transcripts

Metrics count distinct values per crop against native PDF text. They are not manually verified accuracy and do not evaluate element association or discrepancies.


In [ ]:
from l2c.ml_benchmark import tokens, quantity_pairs
for backend in ("paddle", "florence"):
    result_path = SAMPLES_DIR / f"{backend}-results.json"
    if not result_path.exists():
        print(backend, ": no inference results yet")
        continue
    report = json.loads(result_path.read_text(encoding="utf-8"))
    bar_matches = bar_total = pair_matches = pair_total = 0
    durations = []
    for sample, result in zip(samples, report["results"]):
        native = " ".join(sample["native_text"])
        a, b = tokens(native), tokens(result["text"])
        qa, qb = quantity_pairs(native), quantity_pairs(result["text"])
        bar_matches += len(a & b); bar_total += len(a)
        pair_matches += len(qa & qb); pair_total += len(qa)
        durations.append(result["seconds"])
    print(backend, "bar matches:", f"{bar_matches}/{bar_total}",
          "quantity-bar matches:", f"{pair_matches}/{pair_total}",
          "mean seconds:", round(sum(durations) / len(durations), 3))


## 5. Required end-to-end stages — pending

1. Extract reinforcement and produce Annex A records with annotation-centre PDF coordinates.
2. Validate records using Pydantic.
3. Match elements between plans and shop drawings; retain unresolved cases.
4. Compare attributes and classify elements without duplicate counting.
5. Validate CLP against its spreadsheet and test on another project.
6. Generate JSON and per-sheet PDF reports.

Once implemented, this notebook and the CLI will call the same run_project function. These stages are not presented as working capabilities.

## Reproducibility

Restart Kernel and Run All. The default runs inventory and crop preparation; ML is opt-in. The saved repository notebook must have empty outputs.
